In [15]:
from pathlib import Path

import ray
import torch

from climanet.tune import run_tune
from climanet.dataset import STDataset
from climanet.predict import predict_monthly_var
from torch.utils.data import random_split

import xarray as xr

In [2]:
data_folder = Path("./eso4clima/dc_data")
run_dir = Path("./runs_daily").resolve()

var_name = "tos"
input_filenames = [data_folder / f"202101_day_ERA5dc_masked_{var_name}.nc", data_folder / f"202102_day_ERA5dc_masked_{var_name}.nc"]
monthly_filenames = [data_folder / f"202101_mon_ERA5dc_full_{var_name}.nc", data_folder / f"202102_mon_ERA5dc_full_{var_name}.nc"]

### Load the data

In [3]:
data_config = {
    "input_filenames": input_filenames,
    "monthly_filenames": monthly_filenames,
    "landmask_filename": data_folder / "era5_lsm_bool.nc",
    "var_name": var_name,
    "patch_size": (1, 40, 40),  # based on the patch_size in model
    "stride": (20, 20),  # data agumentation by overlapping patches
}

In [4]:
input_data = xr.open_mfdataset(data_config["input_filenames"])
monthly_data = xr.open_mfdataset(data_config["monthly_filenames"])
lsm_mask = xr.open_dataset(data_config["landmask_filename"])

### prepare data for tuning

In [5]:
# coordinates of subset
lon_subset = slice(-50, 50)  # one lon -179.9 is nan, check data
lat_subset = slice(-30, 10)

input_data = input_data.sel(lon=lon_subset, lat=lat_subset)
monthly_data = monthly_data.sel(lon=lon_subset, lat=lat_subset)
lsm_mask = lsm_mask.sel(lon=lon_subset, lat=lat_subset) 

# calculate residuals as target
input_data_averaged = input_data.resample(time="MS").mean(skipna=True)
input_data_averaged["time"] = monthly_data["time"]

# Residuals
monthly_data_res = monthly_data - input_data_averaged

var_name = data_config["var_name"]

dataset = STDataset(
    input_da=input_data[var_name],
    monthly_da=monthly_data_res[var_name],
    land_mask=lsm_mask["lsm"],
    patch_size=data_config["patch_size"],
    stride=data_config["stride"],
    sh_embed_dim=96,
    sh_order_L=10,
    is_hourly=False,

)

Patch grid (m x i x j): 2 x 7 x 19 = 266 patches
Overlap: 20 pixels (height), 20 pixels (width)


In [6]:
# create train test data
generator = torch.Generator().manual_seed(42)
train_size = int(0.6 * len(dataset))
validation_size = int(0.3 * len(dataset))
test_size = len(dataset) - train_size - validation_size
train_dataset, validation_dataset, test_dataset = random_split(dataset, [train_size, validation_size, test_size], generator=generator)
print(len(train_dataset), len(validation_dataset), len(test_dataset))

159 79 28


### config for hyper parameter tuning

In [7]:
static_args = {
    "max_num_epochs": 1,
    "num_trials": 1,
    "cpu_per_trial": 2,
    "gpu_per_trial": 0,
    "run_dir": run_dir,
    "device": "cpu",
    "dataloader_num_workers": 2,
    "train_dataset": ray.put(train_dataset),
    "validation_dataset": ray.put(validation_dataset),
    "num_epoch": 1,
    "max_concurrent_trials": 2
}

# parameters to tune
tune_config = {
    "patch_size": ray.tune.grid_search([2, 4]),
    "overlap": 2,
    "embed_dim": 32,
    "dropout": 0.0,
    "hidden": 32,
    "spatial_depth": 3,
    "spatial_heads": 4,
    "optimizer_lr": 1e-1,
    "batch_config": {"batch_size": 10, "accumulation_steps": 5},
}

2026-07-23 10:11:28,180	INFO worker.py:2024 -- Started a local Ray instance.
(_train pid=40937) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/sarah/GitHub/ClimaNet/notebooks/runs_daily/_train_2026-07-23_10-11-56/_train_2bdd9_00001_1_patch_size=4_2026-07-23_10-11-56/checkpoint_000000)


### Run ray tune

In [8]:
results = run_tune(tune_config, static_args)

ray.shutdown()

2026-07-23 10:12:08,344	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/sarah/GitHub/ClimaNet/notebooks/runs_daily/_train_2026-07-23_10-11-56' in 0.0033s.
2026-07-23 10:12:08,348	INFO tune.py:1039 -- Total run time: 11.79 seconds (11.75 seconds for the tuning loop).
(_train pid=40938) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/sarah/GitHub/ClimaNet/notebooks/runs_daily/_train_2026-07-23_10-11-56/_train_2bdd9_00000_0_patch_size=2_2026-07-23_10-11-56/checkpoint_000000)


### Inspect the output

In [9]:
results.get_best_result("loss", "min")

Result(
  metrics={'loss': 0.36547836661338806},
  path='/home/sarah/GitHub/ClimaNet/notebooks/runs_daily/_train_2026-07-23_10-11-56/_train_2bdd9_00001_1_patch_size=4_2026-07-23_10-11-56',
  filesystem='local',
  checkpoint=Checkpoint(filesystem=local, path=/home/sarah/GitHub/ClimaNet/notebooks/runs_daily/_train_2026-07-23_10-11-56/_train_2bdd9_00001_1_patch_size=4_2026-07-23_10-11-56/checkpoint_000000)
)

### Check best model

In [18]:
# find the path to best model
if not ray.is_initialized():
    ray.init()

experiment_path = results.experiment_path  # or add the path above manually as Path("./runs_daily/_train_2026-07-23_10-11-56\").resolve()"
analysis = ray.tune.ExperimentAnalysis(experiment_path)
best_result = analysis.get_best_trial("loss", "min")
best_checkpoint = best_result.checkpoint
model_path = Path(best_checkpoint.path) / "checkpoint.pt"

In [19]:
batch_size = 10
device = "cpu"
dataloader_num_workers = 2


_, test_loss = predict_monthly_var(
    model_path,
    test_dataset,
    batch_size=batch_size,
    device=device,
    return_numpy=False,
    save_predictions=False,
    return_loss=True,
    verbose=False,
    run_dir=run_dir,
    dataloader_num_workers=dataloader_num_workers,
)

In [20]:
test_loss

0.3733350435892741